# Predicting New York City Airbnb Prices with Linear Regression
**Dataset:** New York City Airbnb Open Data (AB_NYC_2019.csv, Kaggle)  
**Objective:** Predict nightly listing price using a Linear Regression model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_style("whitegrid")
RANDOM_STATE = 42

## Step 1: Exploratory Data Analysis

### Task 1.1: Load data, shape, columns, data types

In [ ]:
df = pd.read_csv("AB_NYC_2019.csv")
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.info()
df.head()

### Task 1.2: Column types

In [ ]:
# Numerical
continuous = ["latitude", "longitude", "price", "reviews_per_month"]
discrete   = ["minimum_nights", "number_of_reviews", "calculated_host_listings_count", "availability_365"]
# Categorical
nominal    = ["neighbourhood_group", "neighbourhood", "room_type"]   # no natural order -> no ordinal columns
# Identifiers / text / date
other      = ["id", "name", "host_id", "host_name", "last_review"]

for label, cols in [("Numerical - continuous", continuous), ("Numerical - discrete", discrete),
                    ("Categorical - nominal", nominal), ("Identifier / text / date", other)]:
    print(f"{label:28s}: {cols}")

### Task 1.3: Missing values

In [ ]:
missing = pd.DataFrame({"Missing Count": df.isnull().sum(),
                        "Missing %": (df.isnull().mean() * 100).round(2)})
missing[missing["Missing Count"] > 0]

## Step 2: Data Cleaning

### Task 2.1: `last_review`
Dropped. About 20% missing, and review activity is already captured by `number_of_reviews` and `reviews_per_month`.

In [ ]:
df = df.drop(columns=["last_review"])

### Task 2.2: `reviews_per_month`

In [ ]:
na_rows = df[df["reviews_per_month"].isnull()]
print("Rows missing reviews_per_month:", len(na_rows))
print("number_of_reviews for those rows:", na_rows["number_of_reviews"].unique())

# Missing only when a listing has 0 reviews -> logical fill value is 0
df["reviews_per_month"] = df["reviews_per_month"].fillna(0)

**Finding:** every missing value belongs to a listing with 0 reviews, so it is filled with 0.

### Task 2.3: Drop identifier columns

In [ ]:
df = df.drop(columns=["id", "name", "host_id", "host_name"])
print("Remaining missing values:", df.isnull().sum().sum())

## Step 3: Outlier Identification & Treatment

### Task 3.1: Visual detection

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
sns.boxplot(x=df["price"], ax=ax[0, 0]);            ax[0, 0].set_title("Price - Boxplot")
sns.histplot(df["price"], bins=50, ax=ax[0, 1]);    ax[0, 1].set_title("Price - Histogram")
sns.boxplot(x=df["minimum_nights"], ax=ax[1, 0]);   ax[1, 0].set_title("Minimum Nights - Boxplot")
sns.histplot(df["minimum_nights"], bins=50, ax=ax[1, 1]); ax[1, 1].set_title("Minimum Nights - Histogram")
plt.tight_layout(); plt.show()
df[["price", "minimum_nights"]].describe()

### Task 3.2: Remove extremes using the 95th percentile

In [ ]:
price_cap  = df["price"].quantile(0.95)
nights_cap = df["minimum_nights"].quantile(0.95)
print(f"95th percentile -> price: ${price_cap:.0f}, minimum_nights: {nights_cap:.0f}")

n_before = len(df)
df = df[(df["price"] <= price_cap) & (df["minimum_nights"] <= nights_cap) & (df["price"] > 0)]
print(f"Rows: {n_before} -> {len(df)} (removed {n_before - len(df)})")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["price"], bins=40, ax=ax[0]);          ax[0].set_title("Price (after treatment)")
sns.histplot(df["minimum_nights"], bins=30, ax=ax[1]); ax[1].set_title("Minimum Nights (after treatment)")
plt.tight_layout(); plt.show()

## Step 4: Feature Selection & Categorical Encoding

### Task 4.1: High cardinality
`neighbourhood` has 200+ unique values. One-hot encoding it would create 200+ columns, so the number of features becomes large relative to the data in each category (Curse of Dimensionality). This makes the model complex, unstable for rare neighbourhoods, and prone to overfitting. It is dropped; `neighbourhood_group` (5 boroughs) is kept.

In [ ]:
print("Unique neighbourhoods:", df["neighbourhood"].nunique())
print("Unique boroughs:", df["neighbourhood_group"].nunique())
df = df.drop(columns=["neighbourhood"])

### Task 4.2: One-Hot Encoding (`drop_first=True` avoids the Dummy Variable Trap)

In [ ]:
df = pd.get_dummies(df, columns=["neighbourhood_group", "room_type"], drop_first=True, dtype=int)
print(list(df.columns))
df.head()

## Step 5: Feature-Target Separation & Train/Test Split

In [ ]:
# Task 5.1
X = df.drop(columns=["price"])
y = df["price"]

# Task 5.2: 80/20 split, fixed seed for reproducibility
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## Step 6: Model Training & Evaluation

In [ ]:
# Task 6.1: Train
model = LinearRegression()
model.fit(X_train, y_train)

# Task 6.2: Predict on test data
y_pred = model.predict(X_test)

# Task 6.3: Metrics
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"R-squared : {r2:.4f}")
print(f"MAE       : ${mae:.2f}")
print(f"MSE       : {mse:.2f}")
print(f"RMSE      : ${rmse:.2f}")

## Step 7: Residual Analysis & Interpretation

### Task 7.1: Actual vs Predicted

In [ ]:
plt.figure(figsize=(6.5, 6.5))
plt.scatter(y_test, y_pred, alpha=0.3)
lim = [0, max(y_test.max(), y_pred.max())]
plt.plot(lim, lim, "r--", label="Perfect prediction (45° line)")
plt.xlabel("Actual Price ($)"); plt.ylabel("Predicted Price ($)")
plt.title("Actual vs Predicted Price"); plt.legend(); plt.show()

### Task 7.2: Residual plot and error distribution

In [ ]:
residuals = y_test - y_pred
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].scatter(y_pred, residuals, alpha=0.3)
ax[0].axhline(0, color="r", linestyle="--")
ax[0].set(xlabel="Predicted Price ($)", ylabel="Residual (Actual - Predicted)", title="Residual Plot")
sns.histplot(residuals, bins=50, ax=ax[1])
ax[1].set(xlabel="Residual ($)", title="Error Distribution")
plt.tight_layout(); plt.show()

### Task 7.3: Cheap vs. expensive listings

In [ ]:
res = pd.DataFrame({"Actual": y_test.values, "Predicted": y_pred})
res["Residual"]  = res["Actual"] - res["Predicted"]
res["Abs_Error"] = res["Residual"].abs()
res["Pct_Error"] = res["Abs_Error"] / res["Actual"] * 100
res["Price_Group"] = pd.qcut(res["Actual"], 4, labels=["Q1 (cheapest)", "Q2", "Q3", "Q4 (most expensive)"])

by_group = res.groupby("Price_Group", observed=True).agg(
    Avg_Price=("Actual", "mean"), MAE=("Abs_Error", "mean"),
    Pct_Error=("Pct_Error", "mean"), Mean_Residual=("Residual", "mean")).round(2)
by_group

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
by_group["MAE"].plot(kind="bar", ax=ax[0], color="steelblue", title="MAE ($) by price group", rot=15)
by_group["Pct_Error"].plot(kind="bar", ax=ax[1], color="darkorange", title="Average % error by price group", rot=15)
plt.tight_layout(); plt.show()

# Data-driven conclusion
cheap, exp = by_group.iloc[0], by_group.iloc[-1]
better_pct = "expensive" if exp["Pct_Error"] < cheap["Pct_Error"] else "cheap"
print(f"Cheapest quartile : MAE ${cheap['MAE']:.0f}, avg error {cheap['Pct_Error']:.0f}%, mean residual ${cheap['Mean_Residual']:.0f}")
print(f"Priciest quartile : MAE ${exp['MAE']:.0f}, avg error {exp['Pct_Error']:.0f}%, mean residual ${exp['Mean_Residual']:.0f}")
print(f"\nIn relative terms the model performs better on {better_pct} listings.")
print("In dollar terms the error is larger on", "expensive" if exp["MAE"] > cheap["MAE"] else "cheap", "listings.")

**Conclusion (Task 7.3):** Read the output above. Expensive listings typically show larger dollar errors and a positive mean residual (under-prediction), because a single linear formula pulls predictions toward the average and our features exclude price drivers such as amenities, size, view and décor. Cheap listings are priced in a narrow band with more uniform features, so the model's guesses are closer in dollars, although it tends to over-predict them.

## Final Summary Report

In [ ]:
summary = f"""
SUMMARY REPORT - NYC Airbnb Price Prediction (Linear Regression)

Data: {len(df):,} typical NYC listings after removing extreme prices (> ${price_cap:.0f}/night)
and long stays (> {nights_cap:.0f} nights). 80% used for training, 20% for testing.

R-squared = {r2:.2f}
The model explains about {r2*100:.0f}% of the differences in nightly prices. The remaining
{100 - r2*100:.0f}% depends on factors not in the data (e.g. quality, amenities, view, photos).

Mean Absolute Error (MAE) = ${mae:.0f}
On average, the predicted price is about ${mae:.0f} away from the real nightly price.
If the model predicts $100, the real price is typically between ${max(100 - mae, 0):.0f} and ${100 + mae:.0f}.

RMSE = ${rmse:.0f}. It is larger than MAE because big mistakes are penalised more heavily.

Where it performs best: relative error is {cheap['Pct_Error']:.0f}% for the cheapest quartile vs
{exp['Pct_Error']:.0f}% for the most expensive quartile; dollar error is ${cheap['MAE']:.0f} vs ${exp['MAE']:.0f}.

Business use: suitable for ballpark price suggestions on typical listings, not for luxury listings.
Adding bedrooms, amenities and photo-quality features would likely improve accuracy.
"""
print(summary)
open("summary_report.txt", "w").write(summary)